# Compte des persnames dans tous les fichiers "peintures"

In [ ]:
import os
from pathlib import Path
import xml.etree.ElementTree as ET
from collections import Counter

base_path = Path("../../../corpus/Peinture/")

counts = Counter()

for file in base_path.glob("*.xml"):
    tree = ET.parse(file)
    root = tree.getroot()

    # Namespace TEI (souvent présent)
    ns = {"tei": "http://www.tei-c.org/ns/1.0"}

    for pers in root.findall(".//tei:objectName", ns):
        ref = pers.get("ref")
        if ref:
            counts[ref] += 1

# affichage des résultats
for ref, n in counts.most_common():
    print(ref, n)


In [ ]:
from lxml import etree
from pathlib import Path
import json

base_path = Path("../../../corpus/IndexPersonnes.xml")
output_path = Path("artists_xml_id.json")

tree = etree.parse(base_path)
root = tree.getroot()

ns = {"tei": "http://www.tei-c.org/ns/1.0"}

ulan_ids = []

for person in root.xpath(".//tei:person", namespaces=ns):
    persname = person.find(".//tei:persName", namespaces=ns)
    
    if persname is not None:
        ref = persname.get("ref", "")
        if "getty.edu" in ref and "ULAN" in ref:
            xml_id = person.get("{http://www.w3.org/XML/1998/namespace}id")
            if xml_id:
                ulan_ids.append(xml_id)

# sauvegarde JSON
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(ulan_ids, f, ensure_ascii=False, indent=2)

In [ ]:
import json
from pathlib import Path
import xml.etree.ElementTree as ET
from collections import Counter

base_path = Path("../../../corpus/Peinture/")

# charger la liste des artistes
with open("artists_xml_id.json", "r", encoding="utf-8") as f:
    artists = set(json.load(f))  # set = plus rapide pour tester l'appartenance

counts = Counter()

ns = {"tei": "http://www.tei-c.org/ns/1.0"}

for file in base_path.glob("*.xml"):
    try:
        tree = ET.parse(file)
        root = tree.getroot()

        for pers in root.findall(".//tei:persName", ns):
            ref = pers.get("ref")
            if ref:
                ref_clean = ref.lstrip("#")  # enlève le #
                if ref_clean in artists:
                    counts[ref_clean] += 1

    except ET.ParseError:
        print(f"Erreur XML dans {file}")

# résultats
for artist, n in counts.most_common():
    print(artist, n)

In [ ]:
from lxml import etree
from pathlib import Path
import json

base_path = Path("../../../corpus/IndexPersonnes.xml")

tree = etree.parse(base_path)
root = tree.getroot()

ns = {"tei": "http://www.tei-c.org/ns/1.0"}

cities = set()
artists = []

for person in root.xpath(".//tei:person", namespaces=ns):

    persname = person.find(".//tei:persName", namespaces=ns)
    if persname is None:
        continue

    ref = persname.get("ref", "")
    if "getty.edu" not in ref or "ULAN" not in ref:
        continue

    xml_id = person.get("{http://www.w3.org/XML/1998/namespace}id")

    place = person.find(".//tei:birth/tei:placeName", namespaces=ns)

    if place is not None and place.text:
        city = place.text.strip()
        cities.add(city)

        artists.append({
            "xml_id": xml_id,
            "birth_place": city
        })

cities_list = sorted(cities)

# sauvegarde villes
with open("ulan_birth_cities.json", "w", encoding="utf-8") as f:
    json.dump(cities_list, f, ensure_ascii=False, indent=2)

# sauvegarde artistes + ville
with open("ulan_artists_birthplace.json", "w", encoding="utf-8") as f:
    json.dump(artists, f, ensure_ascii=False, indent=2)

In [ ]:
from lxml import etree
from pathlib import Path
import csv
import re

base_path = Path("../../../corpus/IndexPersonnes.xml")
output_csv = Path("artists_wikidata_ulan.csv")

tree = etree.parse(base_path)
root = tree.getroot()

ns = {"tei": "http://www.tei-c.org/ns/1.0"}

rows = []

for person in root.xpath(".//tei:person", namespaces=ns):

    # wikidata
    source = person.get("source", "")
    if "wikidata.org" not in source:
        continue

    wikidata_id = source.split("/")[-1]

    # ULAN
    persname = person.find(".//tei:persName", namespaces=ns)
    if persname is None:
        continue

    ref = persname.get("ref", "")
    if "getty.edu" not in ref or "ULAN" not in ref:
        continue

    # extraction ULAN numeric id
    match = re.search(r"subjectid=(\d+)", ref)
    ulan_id = match.group(1) if match else ""

    xml_id = person.get("{http://www.w3.org/XML/1998/namespace}id")

    rows.append([xml_id, wikidata_id, ulan_id])

# écriture CSV
with open(output_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["xml_id", "wikidata_id", "ulan_id"])
    writer.writerows(rows)

print(f"{len(rows)} artistes exportés")

In [ ]:
import pandas as pd
import requests
import time
import csv
from pathlib import Path

input_csv = "artists_wikidata_ulan.csv"
output_csv = Path("artists_wikidata_places.csv")

df = pd.read_csv(input_csv)

def get_qid(uri):
    if uri:
        return uri.split("/")[-1]
    return ""

# récupération des wikidata_id déjà traités
if output_csv.exists():
    processed = pd.read_csv(output_csv)["wikidata_id"].tolist()
else:
    processed = []

url = "https://query.wikidata.org/sparql"

# ouverture du CSV en mode append
with open(output_csv, "a", newline="", encoding="utf-8") as f:

    writer = csv.writer(f)

    # si fichier vide, écrire l'en-tête
    if not output_csv.exists() or output_csv.stat().st_size == 0:
        writer.writerow([
            "xml_id",
            "wikidata_id",
            "ulan_id",
            "artist_label",
            "birth_place",
            "birth_place_qid",
            "region1",
            "region1_qid",
            "region2",
            "region2_qid",
            "country",
            "country_qid"
        ])

    for i, row in df.iterrows():

        qid = row["wikidata_id"]

        if qid in processed:
            print(f"{i+1}/{len(df)} → {qid} déjà traité, skip")
            continue

        print(f"{i+1}/{len(df)} → {qid} traitement en cours")

        query = f"""
        SELECT ?artistLabel 
            ?birthPlace ?birthPlaceLabel
            ?country ?countryLabel
            ?region1 ?region1Label
            ?region2 ?region2Label
        WHERE {{
        BIND(wd:{qid} AS ?artist)

        OPTIONAL {{ ?artist wdt:P19 ?birthPlace. }}
        OPTIONAL {{ ?birthPlace wdt:P17 ?country. }}
        OPTIONAL {{ ?birthPlace wdt:P131 ?region1. }}
        OPTIONAL {{ ?region1 wdt:P131 ?region2. }}

        SERVICE wikibase:label {{
            bd:serviceParam wikibase:language "[AUTO_LANGUAGE],fr,en".
        }}
        }}
        """

        try:
            r = requests.get(
                url,
                params={"query": query, "format": "json"},
                headers={"User-Agent": "art-history-research"},
                timeout=60
            )
            data = r.json()
            bindings = data["results"]["bindings"]

            if bindings:
                item = bindings[0]
                artist_label = item.get("artistLabel", {}).get("value", "")
                birth_place = item.get("birthPlaceLabel", {}).get("value", "")
                country = item.get("countryLabel", {}).get("value", "")
                birth_place_uri = item.get("birthPlace", {}).get("value", "")
                country_uri = item.get("country", {}).get("value", "")

                region1 = item.get("region1Label", {}).get("value", "")
                region2 = item.get("region2Label", {}).get("value", "")
                region1_uri = item.get("region1", {}).get("value", "")
                region2_uri = item.get("region2", {}).get("value", "")

                birth_place_qid = get_qid(birth_place_uri)
                country_qid = get_qid(country_uri)
                region1_qid = get_qid(region1_uri)
                region2_qid = get_qid(region2_uri)

            else:
                artist_label = ""
                birth_place = ""
                country = ""
                region1 = ""
                region2 = ""
                birth_place_qid = ""
                region1_qid = ""
                region2_qid = ""
                country_qid = ""


        except Exception as e:
            print("Erreur :", e)
            artist_label = ""
            birth_place = ""
            country = ""
            region1 = ""
            region2 = ""
            birth_place_qid = ""
            region1_qid = ""
            region2_qid = ""
            country_qid = ""

        writer.writerow([
            row["xml_id"],
            qid,
            row["ulan_id"],
            artist_label,
            birth_place,
            birth_place_qid,
            region1,
            region1_qid,
            region2,
            region2_qid,
            country,
            country_qid
        ])

        f.flush()
        time.sleep(1.5)

print("Terminé :", output_csv)

\w*,Q\d*,\d*,,,\n

# Ajouter les QID Manquants

In [ ]:
import csv
import requests
import time

SPARQL_URL = "https://query.wikidata.org/sparql"

cache = {}

def get_region_qid(region_name):
    # vérifier le cache
    if region_name in cache:
        return cache[region_name]

    query = f"""
    SELECT ?region WHERE {{
      ?region rdfs:label "{region_name}"@fr .
      ?region wdt:P31/wdt:P279* wd:Q82794 .
    }}
    LIMIT 1
    """

    headers = {
        "Accept": "application/sparql-results+json",
        "User-Agent": "region-qid-script/1.0"
    }

    r = requests.get(SPARQL_URL, params={"query": query}, headers=headers)
    data = r.json()

    results = data["results"]["bindings"]

    if results:
        uri = results[0]["region"]["value"]
        qid = uri.split("/")[-1]

        cache[region_name] = qid
        return qid

    cache[region_name] = None
    return None


input_file = "personnes_régions_alignées.csv"
output_file = "output.csv"

with open(input_file, newline="", encoding="utf-8") as infile, \
     open(output_file, "w", newline="", encoding="utf-8") as outfile:

    reader = csv.DictReader(infile)
    writer = csv.DictWriter(outfile, fieldnames=reader.fieldnames)

    writer.writeheader()

    for row in reader:
        if not row["region_qid"]:
            region = row["region"]
            if not region:
                continue
            qid = get_region_qid(region)

            if qid:
                row["region_qid"] = qid
                print(f"{region} -> {qid}")

            time.sleep(0.5)

        writer.writerow(row)

print("Cache utilisé :")
for k, v in cache.items():
    print(k, "->", v)

In [ ]:
import csv
import requests
import time

SPARQL_URL = "https://query.wikidata.org/sparql"

cache = {}

def get_coordinates(qid):
    if qid in cache:
        return cache[qid]

    query = f"""
    SELECT ?coord WHERE {{
      wd:{qid} wdt:P625 ?coord .
    }}
    LIMIT 1
    """

    headers = {
        "Accept": "application/sparql-results+json",
        "User-Agent": "region-coordinates-script/1.0"
    }

    r = requests.get(SPARQL_URL, params={"query": query}, headers=headers)
    data = r.json()

    results = data["results"]["bindings"]

    if results:
        coord = results[0]["coord"]["value"]
        coord = coord.replace("Point(", "").replace(")", "")
        lon, lat = coord.split()
        cache[qid] = (lat, lon)
        return lat, lon

    cache[qid] = (None, None)
    return None, None


input_file = "output.csv"
output_file = "output_with_coords.csv"

with open(input_file, newline="", encoding="utf-8") as infile, \
     open(output_file, "w", newline="", encoding="utf-8") as outfile:

    reader = csv.DictReader(infile)

    fieldnames = reader.fieldnames + ["latitude", "longitude"]
    writer = csv.DictWriter(outfile, fieldnames=fieldnames)

    writer.writeheader()

    for row in reader:
        qid = row["region_qid"]

        lat, lon = get_coordinates(qid)

        row["latitude"] = lat
        row["longitude"] = lon

        print(qid, lat, lon)

        writer.writerow(row)

        time.sleep(0.3)

print("Cache utilisé :", cache)

In [ ]:
import json
import csv
from pathlib import Path
import xml.etree.ElementTree as ET
from collections import Counter

base_path = Path("../../../corpus/Peinture/")

# charger la liste des artistes
with open("artists_xml_id.json", "r", encoding="utf-8") as f:
    artists = set(json.load(f))

ns = {"tei": "http://www.tei-c.org/ns/1.0"}

output_file = "artists_by_document.csv"

with open(output_file, "w", newline="", encoding="utf-8") as out:
    writer = csv.writer(out)
    writer.writerow(["document_xml", "artist_xml_id", "count"])

    for file in base_path.glob("*.xml"):
        try:
            tree = ET.parse(file)
            root = tree.getroot()

            counts = Counter()

            for pers in root.findall(".//tei:persName", ns):
                ref = pers.get("ref")
                if ref:
                    ref_clean = ref.lstrip("#")
                    if ref_clean in artists:
                        counts[ref_clean] += 1

            # écrire une ligne par artiste trouvé dans ce document
            for artist, n in counts.items():
                writer.writerow([file.name, artist, n])

        except ET.ParseError:
            print(f"Erreur XML dans {file}")